In [11]:
# inciso 1
import os
import sys
import time
import subprocess
import importlib.util
import numpy as np
import pandas as pd

if importlib.util.find_spec("pyreadr") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "pyreadr"])

import pyreadr

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score

RANDOM_STATE = 123
TEST_SIZE = 0.30

ruta_bd = "data/listings.RData"

objetos = pyreadr.read_r(ruta_bd)
listings = objetos["listings"].copy()

listings["price_num"] = (
    listings["price"]
    .astype(str)
    .str.replace("$", "", regex=False)
    .str.replace(",", "", regex=False)
)
listings["price_num"] = pd.to_numeric(listings["price_num"], errors="coerce")

columnas_numericas = [
    "latitude",
    "longitude",
    "accommodates",
    "bathrooms",
    "bedrooms",
    "beds",
    "minimum_nights",
    "maximum_nights",
    "minimum_minimum_nights",
    "maximum_minimum_nights",
    "minimum_maximum_nights",
    "maximum_maximum_nights",
    "minimum_nights_avg_ntm",
    "maximum_nights_avg_ntm",
    "availability_30",
    "availability_60",
    "availability_90",
    "availability_365",
    "number_of_reviews",
    "number_of_reviews_ltm",
    "number_of_reviews_l30d",
    "availability_eoy",
    "number_of_reviews_ly",
    "estimated_occupancy_l365d",
    "review_scores_rating",
    "review_scores_accuracy",
    "review_scores_cleanliness",
    "review_scores_checkin",
    "review_scores_communication",
    "review_scores_location",
    "review_scores_value",
    "calculated_host_listings_count",
    "calculated_host_listings_count_entire_homes",
    "calculated_host_listings_count_private_rooms",
    "calculated_host_listings_count_shared_rooms",
    "reviews_per_month"
]

columnas_categoricas = [
    "source",
    "host_response_time",
    "host_is_superhost",
    "host_has_profile_pic",
    "host_identity_verified",
    "neighbourhood_cleansed",
    "neighbourhood_group_cleansed",
    "property_type",
    "room_type",
    "has_availability",
    "instant_bookable",
    "city"
]

columnas_numericas = [col for col in columnas_numericas if col in listings.columns]
columnas_categoricas = [col for col in columnas_categoricas if col in listings.columns]
columnas_modelo = columnas_numericas + columnas_categoricas

base_modelo = listings[columnas_modelo + ["price_num", "id"]].copy()
base_modelo = base_modelo.dropna(subset=["price_num"])
base_modelo = base_modelo[base_modelo["price_num"] > 0].copy()

for col in columnas_numericas:
    base_modelo[col] = pd.to_numeric(base_modelo[col], errors="coerce")

base_modelo[columnas_numericas] = base_modelo[columnas_numericas].replace([np.inf, -np.inf], np.nan)

q1, q2 = base_modelo["price_num"].quantile([1/3, 2/3])
base_modelo["categoria_precio"] = np.select(
    [base_modelo["price_num"] <= q1, base_modelo["price_num"] <= q2],
    ["barata", "media"],
    default="cara"
)

X = base_modelo[columnas_modelo]
y = base_modelo["categoria_precio"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y
)

id_train = base_modelo.loc[X_train.index, "id"]
id_test = base_modelo.loc[X_test.index, "id"]

X_train.shape, X_test.shape, y_train.value_counts(), y_test.value_counts()

((53372, 48),
 (22874, 48),
 categoria_precio
 barata    17982
 cara      17783
 media     17607
 Name: count, dtype: int64,
 categoria_precio
 barata    7707
 cara      7621
 media     7546
 Name: count, dtype: int64)

In [12]:
# inciso 2
variable_respuesta = "categoria_precio"
orden_categorias = ["barata", "media", "cara"]
label_encoder = LabelEncoder()
y_train_cod = label_encoder.fit_transform(y_train)
y_test_cod = label_encoder.transform(y_test)

resumen_variable_respuesta = pd.DataFrame({
    "categoria": orden_categorias,
    "train": y_train.value_counts().reindex(orden_categorias).values,
    "test": y_test.value_counts().reindex(orden_categorias).values
})

resumen_variable_respuesta

,categoria,train,test
0,barata,17982,7707
1,media,17607,7546
2,cara,17783,7621


In [13]:
# inciso 3
def crear_preprocesador():
    transformador_numerico = Pipeline(steps=[
        ("imputador", SimpleImputer(strategy="median")),
        ("escalador", StandardScaler(with_mean=False))
    ])
    try:
        codificador = OneHotEncoder(handle_unknown="ignore", min_frequency=20)
    except TypeError:
        codificador = OneHotEncoder(handle_unknown="ignore")
    transformador_categorico = Pipeline(steps=[
        ("imputador", SimpleImputer(strategy="most_frequent")),
        ("codificador", codificador)
    ])
    return ColumnTransformer(transformers=[
        ("numericas", transformador_numerico, columnas_numericas),
        ("categoricas", transformador_categorico, columnas_categoricas)
    ])

modelo_rna_1 = Pipeline(steps=[
    ("preprocesador", crear_preprocesador()),
    ("rna", MLPClassifier(
        hidden_layer_sizes=(64, 32),
        activation="relu",
        solver="adam",
        alpha=0.0001,
        learning_rate_init=0.001,
        max_iter=100,
        early_stopping=True,
        n_iter_no_change=10,
        random_state=RANDOM_STATE
    ))
])

modelo_rna_2 = Pipeline(steps=[
    ("preprocesador", crear_preprocesador()),
    ("rna", MLPClassifier(
        hidden_layer_sizes=(128, 64, 32),
        activation="tanh",
        solver="adam",
        alpha=0.001,
        learning_rate_init=0.0005,
        max_iter=100,
        early_stopping=True,
        n_iter_no_change=10,
        random_state=RANDOM_STATE
    ))
])

topologias = pd.DataFrame([
    {"modelo": "RNA 1", "capas_ocultas": "64, 32", "activacion": "relu"},
    {"modelo": "RNA 2", "capas_ocultas": "128, 64, 32", "activacion": "tanh"}
])

tiempo_inicio = time.time()
modelo_rna_1.fit(X_train, y_train_cod)
tiempo_modelo_1 = time.time() - tiempo_inicio

tiempo_inicio = time.time()
modelo_rna_2.fit(X_train, y_train_cod)
tiempo_modelo_2 = time.time() - tiempo_inicio

topologias["tiempo_entrenamiento_segundos"] = [tiempo_modelo_1, tiempo_modelo_2]
topologias

c:\Users\ninan\AppData\Local\Programs\Python\Python311\Lib\site-packages\numpy\core\fromnumeric.py:86: RuntimeWarning: invalid value encountered in reduce
  return ufunc.reduce(obj, axis, dtype, out, **passkwargs)
c:\Users\ninan\AppData\Local\Programs\Python\Python311\Lib\site-packages\numpy\core\fromnumeric.py:86: RuntimeWarning: invalid value encountered in reduce
  return ufunc.reduce(obj, axis, dtype, out, **passkwargs)
c:\Users\ninan\AppData\Local\Programs\Python\Python311\Lib\site-packages\numpy\core\fromnumeric.py:86: RuntimeWarning: invalid value encountered in reduce
  return ufunc.reduce(obj, axis, dtype, out, **passkwargs)
c:\Users\ninan\AppData\Local\Programs\Python\Python311\Lib\site-packages\numpy\core\fromnumeric.py:86: RuntimeWarning: invalid value encountered in reduce
  return ufunc.reduce(obj, axis, dtype, out, **passkwargs)


,modelo,capas_ocultas,activacion,tiempo_entrenamiento_segundos
0,RNA 1,"64, 32",relu,43.552711
1,RNA 2,"128, 64, 32",tanh,99.588837


In [14]:
# inciso 4
prediccion_modelo_1_cod = modelo_rna_1.predict(X_test)
prediccion_modelo_2_cod = modelo_rna_2.predict(X_test)

prediccion_modelo_1 = label_encoder.inverse_transform(prediccion_modelo_1_cod)
prediccion_modelo_2 = label_encoder.inverse_transform(prediccion_modelo_2_cod)

resultados_prediccion = pd.DataFrame({
    "id": id_test.values,
    "categoria_real": y_test.values,
    "prediccion_rna_1": prediccion_modelo_1,
    "prediccion_rna_2": prediccion_modelo_2
})

resumen_prediccion = pd.DataFrame([
    {"modelo": "RNA 1", "accuracy_test": accuracy_score(y_test_cod, prediccion_modelo_1_cod)},
    {"modelo": "RNA 2", "accuracy_test": accuracy_score(y_test_cod, prediccion_modelo_2_cod)}
])

resultados_prediccion.to_csv("predicciones_clasificacion_rna_inciso_4.csv", index=False)

resumen_prediccion

c:\Users\ninan\AppData\Local\Programs\Python\Python311\Lib\site-packages\numpy\core\fromnumeric.py:86: RuntimeWarning: invalid value encountered in reduce
  return ufunc.reduce(obj, axis, dtype, out, **passkwargs)
c:\Users\ninan\AppData\Local\Programs\Python\Python311\Lib\site-packages\numpy\core\fromnumeric.py:86: RuntimeWarning: invalid value encountered in reduce
  return ufunc.reduce(obj, axis, dtype, out, **passkwargs)


,modelo,accuracy_test
0,RNA 1,0.742546
1,RNA 2,0.746830


In [ ]:
# inciso 4
from sklearn.metrics import classification_report
prediccion_modelo_1_cod = modelo_rna_1.predict(X_test)
prediccion_modelo_2_cod = modelo_rna_2.predict(X_test)

prediccion_modelo_1 = label_encoder.inverse_transform(prediccion_modelo_1_cod)
prediccion_modelo_2 = label_encoder.inverse_transform(prediccion_modelo_2_cod)

resultados_prediccion = pd.DataFrame({
    "id": id_test.values,
        "categoria_real": y_test.values,
            "prediccion_rna_1": prediccion_modelo_1,
                "prediccion_rna_2": prediccion_modelo_2
                })

                resumen_prediccion = pd.DataFrame([
                    {"modelo": "RNA 1", "accuracy_test": accuracy_score(y_test_cod, prediccion_modelo_1_cod)},
                        {"modelo": "RNA 2", "accuracy_test": accuracy_score(y_test_cod, prediccion_modelo_2_cod)}
                        ])

                        resultados_prediccion.to_csv("predicciones_clasificacion_rna_inciso_4.csv", index=False)

                        resumen_prediccion
                        print("=== Modelo RNA 1 ===")
                        print(classification_report(y_test_cod, prediccion_modelo_1_cod, target_names=label_encoder.classes_))

                        print("\n=== Modelo RNA 2 ===")
                        print(classification_report(y_test_cod, prediccion_modelo_2_cod, target_names=label_encoder.classes_))

In [ ]:
#inciso 5
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

cm_modelo_1 = confusion_matrix(y_test_cod, prediccion_modelo_1_cod)
cm_modelo_2 = confusion_matrix(y_test_cod, prediccion_modelo_2_cod)

disp1 = ConfusionMatrixDisplay(
    confusion_matrix=cm_modelo_1,
    display_labels=label_encoder.classes_
)
disp1.plot()
plt.title("Matriz de Confusión - RNA 1")
plt.show()

disp2 = ConfusionMatrixDisplay(
    confusion_matrix=cm_modelo_2,
    display_labels=label_encoder.classes_
)
disp2.plot()
plt.title("Matriz de Confusión - RNA 2")
plt.show()




print("Matriz RNA 1")
print(cm_modelo_1)

print("\nMatriz RNA 2")
print(cm_modelo_2)

In [ ]:
##inciso 6
print(resumen_prediccion)
print(topologias)

In [ ]:
#inciso 7 
from sklearn.metrics import accuracy_score

# Accuracy en entrenamiento
acc_train_1 = accuracy_score(y_train_cod, modelo_rna_1.predict(X_train))
acc_train_2 = accuracy_score(y_train_cod, modelo_rna_2.predict(X_train))

# Accuracy en prueba (ya lo tienes, pero lo reutilizamos)
acc_test_1 = accuracy_score(y_test_cod, prediccion_modelo_1_cod)
acc_test_2 = accuracy_score(y_test_cod, prediccion_modelo_2_cod)

comparacion_overfitting = pd.DataFrame([
    {"modelo": "RNA 1", "train": acc_train_1, "test": acc_test_1},
    {"modelo": "RNA 2", "train": acc_train_2, "test": acc_test_2}
])

comparacion_overfitting

In [ ]:
#inciso 8 
#mejor modelo: RNA 2
modelo_rna_2_tuneado = Pipeline(steps=[
    ("preprocesador", crear_preprocesador()),
    ("rna", MLPClassifier(
        hidden_layer_sizes=(128, 64, 32),
        activation="tanh",
        solver="adam",
        alpha=0.0005,             
        learning_rate_init=0.0008, 
        max_iter=200,              
        early_stopping=True,
        n_iter_no_change=10,
        random_state=RANDOM_STATE
    ))
])

# Entrenar
modelo_rna_2_tuneado.fit(X_train, y_train_cod)

# Evaluar
pred_tuneado = modelo_rna_2_tuneado.predict(X_test)

from sklearn.metrics import accuracy_score

acc_tuneado = accuracy_score(y_test_cod, pred_tuneado)

acc_tuneado